# Pokémon TCG AI Battle Challenge

## PTCG-AI Deck Intelligence Strategy Engine

This notebook develops a data-driven **Deck Intelligence and Strategy Engine** for analyzing the Pokémon Trading Card Game competition card pool.

The objective is to transform raw card and attack data into interpretable strategic signals that can be used to evaluate individual cards, identify strong card relationships, and rank promising deck candidates.

## Analytical Pipeline

**Card Data → Data Quality → Feature Engineering → Attack Intelligence → Strategic Card Scoring → Card Synergy → Type-Neutral Synergy → Combat + Synergy Integration → Candidate Ranking → Strategy Visualization → Final Validation**

## Tasks Performed

### 1. Data Discovery & Preparation
* Discover and load the competition card dataset.
* Inspect dataset structure, schema, missing values, and available card attributes.
* Isolate the Pokémon card pool for strategic analysis.
* Prepare attack-level and card-level analytical datasets.

### 2. Attack & Combat Intelligence
* Parse energy requirements from attack costs.
* Extract numerical damage values from attack descriptions.
* Calculate **Damage-per-Energy (DPE)**.
* Analyze damage distributions and energy-cost distributions.
* Identify high-damage and energy-efficient attacks.
* Construct card-level offensive profiles.

### 3. Strategic Card Intelligence
Cards are evaluated using multiple measurable dimensions, including:
* Offensive efficiency
* Survivability
* Mobility
* Evolution accessibility
* Strategic utility
These features are combined into an interpretable **Strategic Card Intelligence Score**.

### 4. Card Synergy Intelligence
The notebook evaluates pairwise card relationships using:
* Type compatibility
* Evolution/name relationships
* Strategic profile similarity
* Pairwise synergy scoring
The analysis then investigates whether the synergy model contains excessive type bias.

### 5. Type-Neutral Synergy
A dedicated diagnostic separates the direct type contribution from the overall synergy score.
This allows the engine to evaluate **strategic compatibility beyond simple same-type relationships**, producing a type-neutral synergy signal for subsequent ranking.

### 6. Integrated Strategic Ranking
Combat strength and type-neutral synergy are combined to produce:
* Strategic combat profiles
* Card synergy profiles
* Integrated combat + synergy scores
* Deck candidate scores
* Final candidate rankings
* Candidate-to-candidate synergy relationships

### 7. Visualization & Validation
The notebook provides graphical and tabular analysis of:
* Damage distributions
* Attack efficiency
* Energy requirements
* Offensive strength
* Utility
* Strategic combat performance
* Synergy relationships
* Candidate rankings
* Final strategy signals

## Final Objective
The resulting framework provides an **interpretable, reproducible foundation for AI-assisted PTCG deck intelligence**.
Rather than relying only on raw attack damage or individual card statistics, the engine combines **combat performance, efficiency, utility, strategic similarity, and card synergy** to identify cards with stronger overall strategic potential.
This version establishes the analytical foundation for future extensions such as full deck optimization, game-state simulation, opponent-aware strategy, and AI action selection.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from itertools import combinations
import random
import math
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Random seed initialized: {RANDOM_SEED}")
# Fallback for display() when running as a standard script
try:
    from IPython.display import display
except ImportError:
    display = print

# Configure professional plotting aesthetics
sns.set_theme(style="whitegrid", context="notebook", palette="crest")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

## Experiment Configuration
The strategic analysis and simulation framework uses centralized configuration parameters to ensure reproducibility and simplify experimentation.
These parameters control deck construction, simulation size, and analytical thresholds without requiring changes to the underlying functions.

In [ ]:
CONFIG = {
    # Pokémon TCG fundamentals
    "deck_size": 60,
    "opening_hand_size": 7,
    "prize_count": 6,

    # Monte Carlo simulation
    "simulation_runs": 50_000,
    # Strategic analysis
    "top_cards": 20,
    "top_archetypes": 10,

    # Reproducibility
    "random_seed": RANDOM_SEED
}

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

for key, value in CONFIG.items():
    print(f"{key:<25}: {value}")

# Dataset Discovery
The competition dataset is inspected to identify the available files and establish the primary card database used throughout the analysis.
The discovered dataset will serve as the foundation for card feature engineering, strategic scoring, deck construction, and simulation.

### Data Cleansing & Parsing Functions
To conduct rigorous analysis, we parse game-mechanic constants:
- **Retreat Costs**: Null values on Pokémon cards represent **Free Retreat (0.0 energy)**.
- **Damage Metrics**: Attacks can deal flat damage (`100`), scaling damage (`30×`), or conditional decreasing damage (`-120`).
- **Attack Energy Costs**: Attacks require specific energy counts (e.g., `{G}●` is 2 energy).

In [ ]:
DATA_DIR = "/kaggle/input"
available_files = []
for root, _, files in os.walk(DATA_DIR):
    for file in files:
        available_files.append(os.path.join(root, file))
print("=" * 60)
print("AVAILABLE KAGGLE DATA FILES")
print("=" * 60)
print(f"Total files discovered: {len(available_files)}\n")
for file in available_files:
    print(file)

## Card Dataset Loading
The English card database is used as the primary analytical dataset because the competition environment and strategy framework are evaluated using the available competition card pool.
The Japanese dataset and card-ID reference documents are retained as supporting competition resources but are not required for the core strategic analysis.

In [ ]:
ENGLISH_CARD_FILES = [
    file for file in available_files
    if os.path.basename(file) == "EN Card Data.csv"
    or os.path.basename(file) == "EN_Card_Data.csv"
]
print("English card datasets found:")
for file in ENGLISH_CARD_FILES:
    print(f"  • {file}")
if not ENGLISH_CARD_FILES:
    raise FileNotFoundError("English card dataset was not found.")
# Prefer the standardized filename when both versions exist
preferred_file = next(
    (
        file for file in ENGLISH_CARD_FILES
        if os.path.basename(file) == "EN_Card_Data.csv"
    ),
    ENGLISH_CARD_FILES[0]
)
print(f"\nSelected dataset:")
print(preferred_file)
df = pd.read_csv(preferred_file)
print("\nDataset loaded successfully.")
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

## Dataset Structure
The loaded card database is inspected to understand its schema and identify the fields available for strategic feature engineering.

In [ ]:
print("=" * 70)
print("PTCG CARD DATASET")
print("=" * 70)

print(f"Shape: {df.shape}")

print("\nColumn names:")
for index, column in enumerate(df.columns, start=1):
    print(f"{index:>3}. {column}")

print("\nFirst 5 records:")
display(df.head())

## Data Quality Summary

In [ ]:
quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})
display(quality_report)

## POKÉMON Strategic Dataset Analysis
Individual card statistics provide useful information, but competitive value depends on multiple interacting characteristics.
This section develops a composite **Strategic Card Intelligence Score** using measurable attributes from the card database.
The score evaluates Pokémon using five strategic dimensions:
1. **Offensive Efficiency** — damage relative to energy requirements.
2. **Survivability** — HP relative to the observed Pokémon pool.
3. **Mobility** — retreat efficiency.
4. **Evolution Accessibility** — whether the Pokémon can enter play without requiring a previous evolution stage.
5. **Strategic Utility** — additional effects and abilities that may provide value beyond raw damage.
The resulting score is used as an interpretable ranking signal for later deck construction, archetype discovery, and simulation.

In [ ]:
# Inspect the values used by the Stage/Type column
stage_type_distribution = (
    df["Stage (Pokémon)/Type (Energy and Trainer)"]
    .fillna("Missing")
    .value_counts()
)
display(stage_type_distribution.to_frame("Count"))

In [ ]:
category_distribution = (
    df["Category"]
    .fillna("Missing")
    .value_counts()
)
display(category_distribution.to_frame("Count"))

In [ ]:
POKEMON_STAGES = [
    "Basic Pokémon",
    "Stage 1 Pokémon",
    "Stage 2 Pokémon"
]

pokemon_df = df[
    df["Stage (Pokémon)/Type (Energy and Trainer)"]
    .isin(POKEMON_STAGES)
].copy()

pokemon_df = pokemon_df.reset_index(drop=True)

print("=" * 60)
print("POKÉMON STRATEGIC ANALYSIS DATASET")
print("=" * 60)

print(f"Rows: {len(pokemon_df):,}")
print(f"Unique card names: {pokemon_df['Card Name'].nunique():,}")

print("\nPokémon Stage Distribution:")
display(
    pokemon_df[
        "Stage (Pokémon)/Type (Energy and Trainer)"
    ]
    .value_counts()
    .to_frame("Count")
)

# ATTACK-LEVEL Feature Preparation

In [ ]:
attack_df = pokemon_df.copy()

attack_df["HP_numeric"] = pd.to_numeric(
    attack_df["HP"],
    errors="coerce"
)

attack_df["Retreat_numeric"] = pd.to_numeric(
    attack_df["Retreat"],
    errors="coerce"
)

print("=" * 60)
print("ATTACK-LEVEL FEATURE PREPARATION")
print("=" * 60)

print(f"Records available: {len(attack_df):,}")

display(
    attack_df[
        [
            "Card Name",
            "Stage (Pokémon)/Type (Energy and Trainer)",
            "HP",
            "Type",
            "Weakness",
            "Resistance (Type)",
            "Retreat",
            "Move Name",
            "Cost",
            "Damage"
        ]
    ].head(10)
)

## ENERGY COST PARSER

In [ ]:
def parse_energy_cost(cost):
    """
    Convert symbolic attack costs into a numerical
    energy requirement.
    Examples:
        {G}{G}   -> 2
        {D}●●     -> 3
        ●●       -> 2
        {G}●     -> 2
    """

    if pd.isna(cost):
        return 0
    cost = str(cost)
    typed_energy = re.findall(r"\{[^}]+\}", cost)
    colorless_energy = re.findall(r"●", cost)
    return len(typed_energy) + len(colorless_energy)
attack_df["Energy_Cost"] = (
    attack_df["Cost"]
    .apply(parse_energy_cost)
)
display(
    attack_df[
        [
            "Card Name",
            "Move Name",
            "Cost",
            "Energy_Cost"
        ]
    ].head(15)
)

## Damage Extraction

In [ ]:
def parse_damage(damage):
    """
    Extract the explicit numerical damage value.
    """
    if pd.isna(damage):
        return 0.0
    match = re.search(r"\d+(?:\.\d+)?", str(damage))
    if match:
        return float(match.group())
    return 0.0
attack_df["Damage_numeric"] = (
    attack_df["Damage"]
    .apply(parse_damage)
)
display(
    attack_df[
        [
            "Card Name",
            "Move Name",
            "Damage",
            "Damage_numeric"
        ]
    ].head(15)
)

## Damage Per Energy

In [ ]:
attack_df["DPE"] = np.where(
    attack_df["Energy_Cost"] > 0,
    attack_df["Damage_numeric"] / attack_df["Energy_Cost"],
    0
)
top_dpe_attacks = (
    attack_df[
        attack_df["Energy_Cost"] > 0
    ]
    .sort_values("DPE", ascending=False)
)
print("=" * 60)
print("TOP ATTACKS BY DAMAGE PER ENERGY")
print("=" * 60)
display(
    top_dpe_attacks[
        [
            "Card Name",
            "Move Name",
            "Cost",
            "Damage_numeric",
            "Energy_Cost",
            "DPE"
        ]
    ].head(20)
)

# Card-Level Strategic Summary

In [ ]:
card_summary = (
    attack_df
    .groupby("Card Name")
    .agg(
        Max_DPE=("DPE", "max"),
        Max_Damage=("Damage_numeric", "max"),
        Avg_DPE=("DPE", "mean"),
        HP=("HP_numeric", "first"),
        Retreat=("Retreat_numeric", "first"),
        Stage=("Stage (Pokémon)/Type (Energy and Trainer)", "first"),
        Type=("Type", "first")
    )
    .reset_index()
)
print("=" * 60)
print("CARD-LEVEL STRATEGIC SUMMARY")
print("=" * 60)
print(f"Unique Pokémon analyzed: {len(card_summary):,}")
display(card_summary.head(15))

## Strategic Feature Normalization

In [ ]:
def min_max_normalize(series):
    min_value = series.min()
    max_value = series.max()
    if max_value == min_value:
        return pd.Series(0.5, index=series.index)
    return (series - min_value) / (max_value - min_value)
card_summary["Offensive_Score"] = min_max_normalize(
    card_summary["Max_DPE"].fillna(0)
)
card_summary["Survivability_Score"] = min_max_normalize(
    card_summary["HP"].fillna(card_summary["HP"].median())
)
card_summary["Mobility_Score"] = 1 - min_max_normalize(
    card_summary["Retreat"].fillna(card_summary["Retreat"].median())
)
display(
    card_summary[
        [
            "Card Name",
            "Max_DPE",
            "HP",
            "Retreat",
            "Offensive_Score",
            "Survivability_Score",
            "Mobility_Score"
        ]
    ].head(15)
)

## Evlution Summary

In [ ]:
stage_scores = {
    "Basic Pokémon": 1.0,
    "Stage 1 Pokémon": 0.6,
    "Stage 2 Pokémon": 0.3
}
card_summary["Evolution_Accessibility"] = (
    card_summary["Stage"]
    .map(stage_scores)
    .fillna(0.5)
)
display(
    card_summary[
        [
            "Card Name",
            "Stage",
            "Evolution_Accessibility"
        ]
    ].head(15)
)


## Strategic Utility
Raw damage does not fully represent a Pokémon's competitive value. Effects and abilities can provide additional strategic utility through disruption, protection, resource generation, status effects, or other gameplay advantages.
A lightweight text-based utility signal is therefore derived from the `Effect Explanation` field.

In [ ]:
utility_keywords = [
    "draw",
    "search",
    "switch",
    "heal",
    "damage",
    "discard",
    "energy",
    "bench",
    "protect",
    "prevent",
    "status",
    "poison",
    "confuse",
    "paralyze",
    "burn",
    "retreat"
]
def calculate_utility_score(effect):
    if pd.isna(effect):
        return 0.0
    text = str(effect).lower()
    matches = sum(
        keyword in text
        for keyword in utility_keywords
    )
    return min(matches / 5, 1.0)
card_summary["Utility_Score"] = (
    card_summary["Card Name"]
    .map(
        attack_df.groupby("Card Name")["Effect Explanation"]
        .apply(
            lambda effects: max(
                (calculate_utility_score(effect) for effect in effects),
                default=0.0
            )
        )
    )
    .fillna(0)
)
display(
    card_summary[
        [
            "Card Name",
            "Utility_Score"
        ]
    ].head(15)
)

## Strategic Intelligence Score

In [ ]:
card_summary["Strategic_Score"] = (
    0.30 * card_summary["Offensive_Score"] +
    0.25 * card_summary["Survivability_Score"] +
    0.15 * card_summary["Mobility_Score"] +
    0.15 * card_summary["Evolution_Accessibility"] +
    0.15 * card_summary["Utility_Score"]
) * 100
card_summary["Strategic_Score"] = (
    card_summary["Strategic_Score"].round(2)
)
top_strategic_cards = (
    card_summary
    .sort_values("Strategic_Score", ascending=False)
    .head(20)
)
display(
    top_strategic_cards[
        [
            "Card Name",
            "Stage",
            "Type",
            "Strategic_Score",
            "Offensive_Score",
            "Survivability_Score",
            "Mobility_Score",
            "Evolution_Accessibility",
            "Utility_Score"
        ]
    ]
)

## Strategic Score Utilization

In [ ]:
print("=" * 60)
print("STRATEGIC SCORE DISTRIBUTION")
print("=" * 60)
print(f"Mean Score   : {card_summary['Strategic_Score'].mean():.2f}")
print(f"Median Score : {card_summary['Strategic_Score'].median():.2f}")
print(f"Minimum Score: {card_summary['Strategic_Score'].min():.2f}")
print(f"Maximum Score: {card_summary['Strategic_Score'].max():.2f}")
plt.figure(figsize=(10, 5))
plt.hist(
    card_summary["Strategic_Score"],
    bins=20
)
plt.title("Distribution of Strategic Intelligence Scores")
plt.xlabel("Strategic Score")
plt.ylabel("Number of Pokémon")
plt.show()

## Top Strategic Cards

In [ ]:
top_10_strategic = (
    card_summary
    .sort_values("Strategic_Score", ascending=False)
    .head(10)
    .sort_values("Strategic_Score")
)
plt.figure(figsize=(10, 6))
plt.barh(
    top_10_strategic["Card Name"],
    top_10_strategic["Strategic_Score"]
)
plt.title("Top 10 Pokémon by Strategic Intelligence Score")
plt.xlabel("Strategic Intelligence Score")
plt.ylabel("Pokémon")
plt.tight_layout()
plt.show()

## Card Synergy Analysis
Individual card strength does not necessarily translate into a strong deck.
Competitive deck construction depends on how cards interact with one another. This section introduces a lightweight card-synergy model based on observable relationships in the card dataset.
Synergy is estimated using shared characteristics such as Pokémon Type, evolution relationships, and compatible strategic profiles.
The resulting synergy signal will later be used to identify compatible card groups and construct candidate archetypes for simulation.

In [ ]:
synergy_df = card_summary.copy()
# Clean type information
synergy_df["Type_Clean"] = (
    synergy_df["Type"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)
# Clean stage information
synergy_df["Stage_Clean"] = (
    synergy_df["Stage"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)
# Extract the base Pokémon name from evolution chains
synergy_df["Base_Name"] = (
    synergy_df["Card Name"]
    .str.replace(r"\s+ex$", "", regex=True)
    .str.strip()
)
print("=" * 60)
print("CARD SYNERGY FEATURE PREPARATION")
print("=" * 60)
print(f"Cards available for synergy analysis: {len(synergy_df):,}")
display(
    synergy_df[
        [
            "Card Name",
            "Type_Clean",
            "Stage_Clean",
            "Base_Name",
            "Strategic_Score"
        ]
    ].head(15)
)

## Type Compatibility

In [ ]:
synergy_df["Type_Compatibility"] = (
    synergy_df["Type_Clean"]
    .map(
        synergy_df["Type_Clean"].value_counts()
    )
)
# Normalize popularity of each type to 0–1
synergy_df["Type_Compatibility"] = (
    synergy_df["Type_Compatibility"] /
    synergy_df["Type_Compatibility"].max()
)
print("=" * 60)
print("TYPE COMPATIBILITY")
print("=" * 60)
display(
    synergy_df[
        [
            "Card Name",
            "Type_Clean",
            "Type_Compatibility"
        ]
    ]
    .sort_values("Type_Compatibility", ascending=False)
    .head(20)
)

## Type Support Density

In [ ]:
type_counts = (
    synergy_df["Type_Clean"]
    .value_counts()
)
synergy_df["Type_Support_Density"] = (
    synergy_df["Type_Clean"]
    .map(type_counts)
    / type_counts.max()
)
print("=" * 60)
print("TYPE SUPPORT DENSITY")
print("=" * 60)
display(
    synergy_df[
        [
            "Card Name",
            "Type_Clean",
            "Type_Support_Density"
        ]
    ]
    .sort_values(
        "Type_Support_Density",
        ascending=False
    )
    .head(20)
)

## Evolution Relationship Signal

In [ ]:
pokemon_names = set(synergy_df["Card Name"])

def find_evolution_match(card_name):
    """
    Check whether the card name appears as a previous-stage
    requirement for another Pokémon in the dataset.
    """

    matches = attack_df[
        attack_df["Previous stage"]
        .fillna("")
        .astype(str)
        .str.contains(
            re.escape(card_name),
            case=False,
            regex=True
        )
    ]

    return len(matches)


synergy_df["Evolution_Link_Count"] = (
    synergy_df["Card Name"]
    .apply(find_evolution_match)
)

max_links = synergy_df["Evolution_Link_Count"].max()

if max_links > 0:
    synergy_df["Evolution_Link_Score"] = (
        synergy_df["Evolution_Link_Count"] / max_links
    )
else:
    synergy_df["Evolution_Link_Score"] = 0.0

print("=" * 60)
print("EVOLUTION RELATIONSHIP SIGNAL")
print("=" * 60)

display(
    synergy_df[
        [
            "Card Name",
            "Evolution_Link_Count",
            "Evolution_Link_Score"
        ]
    ]
    .sort_values(
        "Evolution_Link_Count",
        ascending=False
    )
    .head(20)
)

## Strategic Profile Similarity

In [ ]:
strategy_features = [
    "Offensive_Score",
    "Survivability_Score",
    "Mobility_Score",
    "Evolution_Accessibility",
    "Utility_Score"
]
synergy_df["Strategic_Profile"] = (
    synergy_df[strategy_features]
    .mean(axis=1)
)
print("=" * 60)
print("STRATEGIC PROFILE SIMILARITY")
print("=" * 60)
display(
    synergy_df[
        [
            "Card Name",
            "Strategic_Profile",
            "Strategic_Score"
        ]
    ]
    .sort_values(
        "Strategic_Profile",
        ascending=False
    )
    .head(20)
)

# Pairwise Card Synergy

In [ ]:
def calculate_pair_synergy(card_a, card_b):
    score = 0.0
    # Same Pokémon Type
    if card_a["Type_Clean"] == card_b["Type_Clean"]:
        score += 0.40
    # Evolution relationship
    name_a = str(card_a["Card Name"]).lower()
    name_b = str(card_b["Card Name"]).lower()
    if name_a in name_b or name_b in name_a:
        score += 0.25
    # Strategic profile similarity
    profile_difference = abs(
        card_a["Strategic_Profile"] -
        card_b["Strategic_Profile"]
    )
    strategic_similarity = max(
        0,
        1 - profile_difference
    )
    score += 0.35 * strategic_similarity
    return score
print("=" * 60)
print("PAIRWISE CARD SYNERGY")
print("=" * 60)
print("Pairwise synergy function initialized.")
print(f"Cards available: {len(synergy_df):,}")

## Generate Complete Candidate Synergy Pairs

In [ ]:
candidate_pairs = []
records = synergy_df.to_dict("records")
for i in range(len(records)):
    card_a = records[i]
    for j in range(i + 1, len(records)):
        card_b = records[j]
        candidate_pairs.append(
            (
                card_a["Card Name"],
                card_b["Card Name"]
            )
        )
print("=" * 60)
print("COMPLETE CANDIDATE SYNERGY PAIRS")
print("=" * 60)
print(f"Cards available: {len(records):,}")
print(f"Candidate pairs generated: {len(candidate_pairs):,}")
print("\nFirst 10 candidate pairs:")
for pair in candidate_pairs[:10]:
    print(f"  • {pair[0]}  ↔  {pair[1]}")

## Calculate Pairwise Synergy Scores

In [ ]:
print("=" * 60)
print("CALCULATING PAIRWISE SYNERGY SCORES")
print("=" * 60)
card_lookup = {
    row["Card Name"]: row
    for row in records
}
synergy_results = []
for card_a_name, card_b_name in candidate_pairs:
    card_a = card_lookup[card_a_name]
    card_b = card_lookup[card_b_name]
    synergy_score = calculate_pair_synergy(
        card_a,
        card_b
    )
    synergy_results.append({
        "Card_A": card_a_name,
        "Card_B": card_b_name,
        "Type_A": card_a["Type_Clean"],
        "Type_B": card_b["Type_Clean"],
        "Strategic_A": card_a["Strategic_Score"],
        "Strategic_B": card_b["Strategic_Score"],
        "Synergy_Score": synergy_score
    })
synergy_pairs_df = pd.DataFrame(synergy_results)
print(f"Scored pairs: {len(synergy_pairs_df):,}")
display(
    synergy_pairs_df
    .sort_values("Synergy_Score", ascending=False)
    .head(20)
)

## Top Strategic Synergy Pairs

In [ ]:
top_synergy_pairs = (
    synergy_pairs_df
    .sort_values("Synergy_Score", ascending=False)
    .head(25)
    .reset_index(drop=True)
)
print("=" * 60)
print("TOP STRATEGIC SYNERGY PAIRS")
print("=" * 60)
display(
    top_synergy_pairs[
        [
            "Card_A",
            "Card_B",
            "Type_A",
            "Type_B",
            "Strategic_A",
            "Strategic_B",
            "Synergy_Score"
        ]
    ]
)

## Synergy Score Distribution

In [ ]:
print("=" * 60)
print("SYNERGY SCORE DISTRIBUTION")
print("=" * 60)
synergy_stats = (
    synergy_pairs_df["Synergy_Score"]
    .describe()
    .round(4)
)
display(synergy_stats.to_frame("Value"))
strong_synergy_threshold = (
    synergy_pairs_df["Synergy_Score"]
    .quantile(0.90)
)
print(
    f"\nStrong synergy threshold (90th percentile): "
    f"{strong_synergy_threshold:.4f}"
)
strong_pairs = synergy_pairs_df[
    synergy_pairs_df["Synergy_Score"] >= strong_synergy_threshold
]
print(
    f"Strong synergy pairs: "
    f"{len(strong_pairs):,}"
)


## Synergy Network  / Hub Analysis

In [ ]:
strong_synergy_pairs = synergy_pairs_df[
    synergy_pairs_df["Synergy_Score"] >= strong_synergy_threshold
].copy()
hub_counts = Counter()
for _, row in strong_synergy_pairs.iterrows():
    hub_counts[row["Card_A"]] += 1
    hub_counts[row["Card_B"]] += 1
synergy_hubs = pd.DataFrame(
    hub_counts.items(),
    columns=["Card Name", "Strong_Synergy_Links"]
)
synergy_hubs = synergy_hubs.merge(
    synergy_df[
        [
            "Card Name",
            "Type_Clean",
            "Stage_Clean",
            "Strategic_Score"
        ]
    ],
    on="Card Name",
    how="left"
)
synergy_hubs = (
    synergy_hubs
    .sort_values(
        ["Strong_Synergy_Links", "Strategic_Score"],
        ascending=False
    )
    .reset_index(drop=True)
)
print("=" * 60)
print("SYNERGY NETWORK / HUB ANALYSIS")
print("=" * 60)
print(
    f"Strong synergy relationships: "
    f"{len(strong_synergy_pairs):,}"
)
print(
    f"Pokémon with at least one strong link: "
    f"{len(synergy_hubs):,}"
)
print("\nTop synergy hubs:")
display(
    synergy_hubs.head(20)
)

## Synergy Hub Score

In [ ]:
max_links = synergy_hubs["Strong_Synergy_Links"].max()
synergy_hubs["Hub_Link_Score"] = (
    synergy_hubs["Strong_Synergy_Links"] / max_links
)
synergy_hubs["Hub_Score"] = (
    0.60 * synergy_hubs["Hub_Link_Score"] +
    0.40 * (synergy_hubs["Strategic_Score"] / 100)
) * 100
synergy_hubs["Hub_Score"] = (
    synergy_hubs["Hub_Score"].round(2)
)
top_synergy_hubs = (
    synergy_hubs
    .sort_values("Hub_Score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
print("=" * 60)
print("TOP SYNERGY HUBS")
print("=" * 60)
display(
    top_synergy_hubs[
        [
            "Card Name",
            "Type_Clean",
            "Stage_Clean",
            "Strong_Synergy_Links",
            "Strategic_Score",
            "Hub_Score"
        ]
    ]
)

## Synergy Neighborhoods

In [ ]:
def get_synergy_neighbors(card_name, top_n=10):
    neighbors = synergy_pairs_df[
        (synergy_pairs_df["Card_A"] == card_name) |
        (synergy_pairs_df["Card_B"] == card_name)
    ].copy()
    neighbors["Neighbor"] = np.where(
        neighbors["Card_A"] == card_name,
        neighbors["Card_B"],
        neighbors["Card_A"]
    )
    return (
        neighbors[
            ["Neighbor", "Synergy_Score"]
        ]
        .sort_values(
            "Synergy_Score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )
# Inspect the strongest synergy hub
hub_card = top_synergy_hubs.iloc[0]["Card Name"]
print("=" * 60)
print("SYNERGY NEIGHBORHOOD")
print("=" * 60)
print(f"Hub card: {hub_card}")
display(
    get_synergy_neighbors(
        hub_card,
        top_n=10
    )
)

## Top Synergy Hub Neighborhoods

In [ ]:
print("=" * 60)
print("TOP SYNERGY HUB NEIGHBORHOODS")
print("=" * 60)
top_hubs = top_synergy_hubs.head(10)["Card Name"].tolist()
for i, hub in enumerate(top_hubs, 1):
    print(f"\n{i}. {hub}")
    neighbors = get_synergy_neighbors(
        hub,
        top_n=5
    )
    display(neighbors)

## Synergy Composition Validation

In [ ]:
print("=" * 60)
print("SYNERGY COMPOSITION VALIDATION")
print("=" * 60)
synergy_pairs_df["Same_Type"] = (
    synergy_pairs_df["Type_A"] ==
    synergy_pairs_df["Type_B"]
)
type_summary = (
    synergy_pairs_df
    .groupby("Same_Type")["Synergy_Score"]
    .agg(["count", "mean", "min", "max"])
    .round(4)
)
display(type_summary)
print("\nType relationship counts:")
display(
    synergy_pairs_df["Same_Type"]
    .value_counts()
    .rename(index={
        True: "Same Type",
        False: "Different Type"
    })
    .to_frame("Count")
)

## Synergy Score Component Analysis

In [ ]:
print("=" * 60)
print("SYNERGY SCORE COMPONENT ANALYSIS")
print("=" * 60)
def synergy_components(card_a, card_b):

    # 1. TYPE COMPATIBILITY
    same_type = (
        card_a["Type_Clean"] == card_b["Type_Clean"]
    )
    type_score = 0.40 if same_type else 0.0

    # 2. EVOLUTION / NAME RELATIONSHIP
    name_a = str(card_a["Card Name"]).lower()
    name_b = str(card_b["Card Name"]).lower()
    name_relation = (
        name_a in name_b
        or name_b in name_a
    )
    evolution_score = 0.25 if name_relation else 0.0

    # 3. STRATEGIC PROFILE SIMILARITY
    profile_difference = abs(
        card_a["Strategic_Profile"]
        - card_b["Strategic_Profile"]
    )
    strategic_similarity = max(
        0,
        1 - profile_difference
    )
    strategic_score = (
        0.35 * strategic_similarity
    )

    # TOTAL
    total_score = (
        type_score
        + evolution_score
        + strategic_score
    )
    return {
        "Type_Component": type_score,
        "Evolution_Component": evolution_score,
        "Strategic_Component": strategic_score,
        "Synergy_Calculated": total_score
    }

# SAMPLE VALIDATION
component_results = []
records = synergy_df.to_dict("records")
for pair in candidate_pairs[:1000]:
    card_a_name, card_b_name = pair
    card_a = next(
        card for card in records
        if card["Card Name"] == card_a_name
    )
    card_b = next(
        card for card in records
        if card["Card Name"] == card_b_name
    )
    components = synergy_components(
        card_a,
        card_b
    )
    component_results.append({
        "Card_A": card_a_name,
        "Card_B": card_b_name,
        **components
    })
component_analysis_df = pd.DataFrame(
    component_results
)

# COMPONENT STATISTICS
print("\nComponent averages:")
display(
    component_analysis_df[
        [
            "Type_Component",
            "Evolution_Component",
            "Strategic_Component",
            "Synergy_Calculated"
        ]
    ]
    .mean()
    .round(4)
    .to_frame("Mean")
)

# COMPONENT CONTRIBUTION

component_means = (
    component_analysis_df[
        [
            "Type_Component",
            "Evolution_Component",
            "Strategic_Component"
        ]
    ]
    .mean()
)
print("\nComponent contribution:")
display(
    component_means
    .round(4)
    .to_frame("Average_Contribution")
)

# VALIDATE AGAINST EXISTING SCORES
comparison = component_analysis_df.copy()
comparison["Existing_Synergy"] = (
    synergy_pairs_df
    .iloc[:len(comparison)]
    ["Synergy_Score"]
    .values
)
comparison["Difference"] = (
    comparison["Synergy_Calculated"]
    - comparison["Existing_Synergy"]
)
print("\nScore validation:")
display(
    comparison[
        [
            "Card_A",
            "Card_B",
            "Synergy_Calculated",
            "Existing_Synergy",
            "Difference"
        ]
    ].head(10)
)
print("\nMaximum absolute difference:")
print(
    round(
        comparison["Difference"].abs().max(),
        8
    )
)

## Synergy Feature Enrichment

In [ ]:
print("=" * 60)
print("SYNERGY FEATURE ENRICHMENT")
print("=" * 60)
synergy_features_df = synergy_pairs_df.copy()
synergy_features_df["Type_Match"] = (
    synergy_features_df["Type_A"]
    == synergy_features_df["Type_B"]
).astype(int)
synergy_features_df["Strategic_Difference"] = (
    synergy_features_df["Strategic_A"]
    - synergy_features_df["Strategic_B"]
).abs()
synergy_features_df["Strategic_Similarity"] = (
    1 - synergy_features_df["Strategic_Difference"] / 100
).clip(0, 1)
synergy_features_df["Name_Relationship"] = (
    synergy_features_df["Card_A"]
    .str.lower()
    .apply(
        lambda x: x
    )
)
print(f"Synergy pairs: {len(synergy_features_df):,}")
print(
    f"Type matches: "
    f"{synergy_features_df['Type_Match'].sum():,}"
)
display(
    synergy_features_df[
        [
            "Card_A",
            "Card_B",
            "Type_Match",
            "Strategic_Difference",
            "Strategic_Similarity",
            "Synergy_Score"
        ]
    ].head(10)
)

## Synergy Redundancy & Correlation Analysis

In [ ]:
print("=" * 60)
print("SYNERGY REDUNDANCY & CORRELATION ANALYSIS")
print("=" * 60)
correlation_features = [
    "Type_Match",
    "Strategic_Difference",
    "Strategic_Similarity",
    "Synergy_Score"
]
correlation_matrix = (
    synergy_features_df[correlation_features]
    .corr()
    .round(4)
)
display(correlation_matrix)
print("\nFeature-to-Synergy correlations:")
synergy_correlations = (
    correlation_matrix["Synergy_Score"]
    .drop("Synergy_Score")
    .sort_values(ascending=False)
    .to_frame("Correlation")
)
display(synergy_correlations)

## Synergy Model Diagnostic

In [ ]:
print("=" * 60)
print("SYNERGY MODEL DIAGNOSTIC")
print("=" * 60)
synergy_by_type = (
    synergy_features_df
    .groupby("Type_Match")["Synergy_Score"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(4)
)
display(synergy_by_type)
print("\nSynergy difference between type groups:")
type_means = (
    synergy_features_df
    .groupby("Type_Match")["Synergy_Score"]
    .mean()
)
same_type_mean = type_means.get(1, 0)
different_type_mean = type_means.get(0, 0)
print(f"Same-type mean:       {same_type_mean:.4f}")
print(f"Different-type mean:  {different_type_mean:.4f}")
print(f"Mean difference:      {same_type_mean - different_type_mean:.4f}")

## Cross-type Synergy Analysis

In [ ]:
print("=" * 60)
print("CROSS-TYPE SYNERGY ANALYSIS")
print("=" * 60)
cross_type_pairs = synergy_features_df[
    synergy_features_df["Type_Match"] == 0
].copy()
print(f"Cross-type pairs: {len(cross_type_pairs):,}")
cross_type_summary = (
    cross_type_pairs["Synergy_Score"]
    .describe()
    .round(4)
    .to_frame("Value")
)
display(cross_type_summary)
print("\nTop cross-type synergy pairs:")
display(
    cross_type_pairs
    .sort_values("Synergy_Score", ascending=False)
    [
        [
            "Card_A",
            "Card_B",
            "Strategic_A",
            "Strategic_B",
            "Strategic_Similarity",
            "Synergy_Score"
        ]
    ]
    .head(20)
)

## Synergy Type BIAS Analysis

In [ ]:
print("=" * 60)
print("SYNERGY TYPE BIAS ANALYSIS")
print("=" * 60)
# Build Type_Match directly from the pairwise type columns
synergy_pairs_enriched = synergy_pairs_df.copy()
synergy_pairs_enriched["Type_Match"] = (
    synergy_pairs_enriched["Type_A"]
    == synergy_pairs_enriched["Type_B"]
)
type_bias = (
    synergy_pairs_enriched
    .groupby("Type_Match")["Synergy_Score"]
    .agg(
        Count="count",
        Mean="mean",
        Std="std",
        Min="min",
        Max="max"
    )
    .round(4)
)
display(type_bias)
same_type_mean = type_bias.loc[True, "Mean"]
cross_type_mean = type_bias.loc[False, "Mean"]
print(f"\nSame-type mean synergy:   {same_type_mean:.4f}")
print(f"Cross-type mean synergy:  {cross_type_mean:.4f}")
print(f"Type bias gap:            {same_type_mean - cross_type_mean:.4f}")

## Type Component Impact

In [ ]:
print("=" * 60)
print("TYPE COMPONENT IMPACT")
print("=" * 60)
type_impact = (
    synergy_pairs_enriched
    .groupby("Type_Match")["Synergy_Score"]
    .agg(
        Count="count",
        Mean="mean",
        Std="std"
    )
)
display(type_impact.round(4))
type_weight = 0.40
print(f"\nConfigured Type Component Weight: {type_weight:.2f}")
print(
    f"Maximum direct Type contribution: "
    f"{type_weight:.2f}"
)
print(
    f"Observed same-type advantage: "
    f"{same_type_mean - cross_type_mean:.4f}"
)

# Type-neutral Synergy Diagnostic

In [ ]:
print("=" * 60)
print("TYPE-NEUTRAL SYNERGY DIAGNOSTIC")
print("=" * 60)
type_neutral_synergy = (
    synergy_pairs_enriched["Synergy_Score"]
    - synergy_pairs_enriched["Type_Match"].astype(float) * 0.40
)
synergy_pairs_enriched["Type_Neutral_Synergy"] = (
    type_neutral_synergy.clip(lower=0)
)
neutral_stats = (
    synergy_pairs_enriched
    .groupby("Type_Match")["Type_Neutral_Synergy"]
    .agg(
        Count="count",
        Mean="mean",
        Std="std",
        Min="min",
        Max="max"
    )
    .round(4)
)
display(neutral_stats)
same_neutral = neutral_stats.loc[True, "Mean"]
cross_neutral = neutral_stats.loc[False, "Mean"]
print(f"\nSame-type neutral mean:   {same_neutral:.4f}")
print(f"Cross-type neutral mean:  {cross_neutral:.4f}")
print(f"Neutral bias gap:         {same_neutral - cross_neutral:.4f}")

## Type-neutral Synergy Score

In [ ]:
print("=" * 60)
print("TYPE-NEUTRAL SYNERGY SCORE")
print("=" * 60)
# Type relationship
synergy_pairs_df["Type_Match"] = (
    synergy_pairs_df["Type_A"]
    == synergy_pairs_df["Type_B"]
)
# Remove the direct Type component from the existing score
synergy_pairs_df["Type_Neutral_Synergy"] = (
    synergy_pairs_df["Synergy_Score"]
    - synergy_pairs_df["Type_Match"].astype(float) * 0.40
)
# Keep values within valid range
synergy_pairs_df["Type_Neutral_Synergy"] = (
    synergy_pairs_df["Type_Neutral_Synergy"]
    .clip(lower=0, upper=1)
)
display(
    synergy_pairs_df[
        [
            "Card_A",
            "Card_B",
            "Type_Match",
            "Type_Neutral_Synergy"
        ]
    ].head(10)
)
print("\nType-neutral synergy statistics:")
display(
    synergy_pairs_df["Type_Neutral_Synergy"]
    .describe()
    .round(4)
    .to_frame("Value")
)

## Neutral Synergy Validation

In [ ]:
print("=" * 60)
print("NEUTRAL SYNERGY VALIDATION")
print("=" * 60)
neutral_validation = (
    synergy_pairs_df
    .groupby("Type_Match")["Type_Neutral_Synergy"]
    .agg(
        Count="count",
        Mean="mean",
        Std="std",
        Min="min",
        Max="max"
    )
    .round(4)
)
display(neutral_validation)
same_type_mean = (
    synergy_pairs_df.loc[
        synergy_pairs_df["Type_Match"] == True,
        "Type_Neutral_Synergy"
    ].mean()
)
cross_type_mean = (
    synergy_pairs_df.loc[
        synergy_pairs_df["Type_Match"] == False,
        "Type_Neutral_Synergy"
    ].mean()
)
neutral_gap = same_type_mean - cross_type_mean
print(f"\nSame-type neutral mean:   {same_type_mean:.4f}")
print(f"Cross-type neutral mean: {cross_type_mean:.4f}")
print(f"Neutral bias gap:         {neutral_gap:.4f}")

## Type-neutral Synergy Ranking

In [ ]:
print("=" * 60)
print("TYPE-NEUTRAL SYNERGY RANKING")
print("=" * 60)
neutral_ranking = (
    synergy_pairs_df
    .groupby("Card_A")["Type_Neutral_Synergy"]
    .agg(
        Mean_Neutral_Synergy="mean",
        Max_Neutral_Synergy="max",
        Synergy_Connections="count"
    )
    .reset_index()
    .sort_values(
        "Mean_Neutral_Synergy",
        ascending=False
    )
)
display(
    neutral_ranking.head(20).round(4)
)
print(
    f"\nCards ranked: "
    f"{len(neutral_ranking):,}"
)

## Card-level Synergy Profile

In [ ]:
print("=" * 60)
print("CARD-LEVEL SYNERGY PROFILE")
print("=" * 60)
card_neutral_profile = (
    synergy_pairs_df
    .groupby("Card_A")["Type_Neutral_Synergy"]
    .agg(
        Mean_Neutral_Synergy="mean",
        Max_Neutral_Synergy="max",
        Synergy_Connections="count"
    )
    .reset_index()
)
card_neutral_profile["Neutral_Synergy_Profile"] = (
    0.70 * card_neutral_profile["Mean_Neutral_Synergy"]
    + 0.30 * card_neutral_profile["Max_Neutral_Synergy"]
)
display(
    card_neutral_profile[
        [
            "Card_A",
            "Mean_Neutral_Synergy",
            "Max_Neutral_Synergy",
            "Synergy_Connections",
            "Neutral_Synergy_Profile"
        ]
    ]
    .sort_values(
        "Neutral_Synergy_Profile",
        ascending=False
    )
    .head(20)
    .round(4)
)
print(
    f"\nCards profiled: "
    f"{len(card_neutral_profile):,}"
)

## Final Card Synergy Score

In [ ]:
print("=" * 60)
print("FINAL CARD SYNERGY SCORE")
print("=" * 60)
final_card_scores = (
    card_summary[
        [
            "Card Name",
            "Strategic_Score"
        ]
    ]
    .merge(
        card_neutral_profile[
            [
                "Card_A",
                "Neutral_Synergy_Profile"
            ]
        ],
        left_on="Card Name",
        right_on="Card_A",
        how="left"
    )
)
final_card_scores["Neutral_Synergy_Profile"] = (
    final_card_scores["Neutral_Synergy_Profile"]
    .fillna(0)
)
# Final score:
# 60% strategic strength
# 40% type-neutral synergy
final_card_scores["Final_Card_Score"] = (
    0.60 * final_card_scores["Strategic_Score"]
    + 0.40 * (
        final_card_scores["Neutral_Synergy_Profile"] * 100
    )
)
final_card_scores = (
    final_card_scores
    .sort_values(
        "Final_Card_Score",
        ascending=False
    )
    .reset_index(drop=True)
)
display(
    final_card_scores[
        [
            "Card Name",
            "Strategic_Score",
            "Neutral_Synergy_Profile",
            "Final_Card_Score"
        ]
    ]
    .head(20)
    .round(4)
)
print(
    f"\nCards scored: "
    f"{len(final_card_scores):,}"
)

## Damage Parsing Utility

In [ ]:
def parse_damage(value):
    """
    Convert raw attack-damage values into a numeric value.
    Returns the maximum numeric damage found.
    Missing or non-damaging values return NaN.
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip()
    if not text:
        return np.nan
    matches = re.findall(r"\d+(?:\.\d+)?", text)
    if not matches:
        return np.nan
    return max(float(value) for value in matches)

# VALIDATION
test_values = [
    120,
    "120",
    "120 damage",
    "50+30",
    "10 / 20 / 30",
    None,
    ""
]
damage_validation = pd.DataFrame({
    "Raw_Value": test_values,
    "Parsed_Damage": [
        parse_damage(value)
        for value in test_values
    ]
})
print("=" * 60)
print("DAMAGE PARSING UTILITY")
print("=" * 60)
display(damage_validation)
print("\nFunction status: READY")

# DAMAGE PARSING VISUALIZATION
plot_data = damage_validation.dropna(
    subset=["Parsed_Damage"]
).copy()
plt.figure(figsize=(9, 5))
plt.barh(
    plot_data["Raw_Value"].astype(str),
    plot_data["Parsed_Damage"]
)
plt.xlabel("Parsed Damage")
plt.ylabel("Raw Attack Value")
plt.title("Damage Parsing Validation")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
print("Parsed_Damage column created:", "Parsed_Damage" in attack_df.columns)

## Damage Distribution Analysis

In [ ]:
print("=" * 60)
print("DAMAGE DISTRIBUTION ANALYSIS")
print("=" * 60)
# Ensure Parsed_Damage exists
if "Parsed_Damage" not in attack_df.columns:
    attack_df["Parsed_Damage"] = (
        attack_df["Damage"]
        .apply(parse_damage)
    )
damage_values = (
    pd.to_numeric(
        attack_df["Parsed_Damage"],
        errors="coerce"
    )
    .dropna()
)
print(f"\nValid damage values: {len(damage_values):,}")
if damage_values.empty:
    print("No valid damage values available for analysis.")
else:
    
    # TEXT ANALYSIS
    damage_stats = damage_values.describe().round(2)
    print("\nDamage statistics:")
    display(damage_stats.to_frame("Value"))
    print("\nDamage percentiles:")
    damage_percentiles = (
        damage_values
        .quantile([0.25, 0.50, 0.75, 0.90, 0.95])
        .round(2)
    )
    display(
        damage_percentiles
        .rename("Damage")
        .to_frame()
    )
    print(
        f"\nMinimum damage: {damage_values.min():.0f}"
    )
    print(
        f"Maximum damage: {damage_values.max():.0f}"
    )
    print(
        f"Mean damage:    {damage_values.mean():.2f}"
    )
    print(
        f"Median damage:  {damage_values.median():.2f}"
    )

    # MATPLOTLIB VISUALIZATION
    plt.figure(figsize=(10, 6))

    plt.hist(
        damage_values,
        bins=30,
        edgecolor="black",
        alpha=0.8
    )
    plt.axvline(
        damage_values.mean(),
        linestyle="--",
        linewidth=2,
        label=f"Mean = {damage_values.mean():.1f}"
    )
    plt.axvline(
        damage_values.median(),
        linestyle=":",
        linewidth=2,
        label=f"Median = {damage_values.median():.1f}"
    )
    plt.title("Attack Damage Distribution")
    plt.xlabel("Parsed Damage")
    plt.ylabel("Number of Attacks")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
print(attack_df.columns.tolist())

## Top Damage Cards

In [ ]:
print("=" * 60)
print("TOP DAMAGE CARDS")
print("=" * 60)
damage_by_card = (
    attack_df
    .assign(
        Parsed_Damage=pd.to_numeric(
            attack_df["Parsed_Damage"],
            errors="coerce"
        )
    )
    .dropna(subset=["Parsed_Damage"])
    .groupby("Card Name")
    .agg(
        Max_Damage=("Parsed_Damage", "max"),
        Mean_Damage=("Parsed_Damage", "mean"),
        Attack_Count=("Parsed_Damage", "count")
    )
    .sort_values(
        ["Max_Damage", "Mean_Damage"],
        ascending=False
    )
)
print(
    f"\nCards with valid damage data: "
    f"{len(damage_by_card):,}"
)
top_damage_cards = (
    damage_by_card
    .head(15)
    .round(2)
)
display(top_damage_cards)

# MATPLOTLIB VISUALIZATION
plot_data = (
    top_damage_cards
    .sort_values("Max_Damage")
)
plt.figure(figsize=(10, 7))
plt.barh(
    plot_data.index,
    plot_data["Max_Damage"],
    edgecolor="black",
    alpha=0.8
)
plt.title("Top Cards by Maximum Attack Damage")
plt.xlabel("Maximum Parsed Damage")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Damage-per-energy (DPE) Analysis

In [ ]:
print("=" * 60)
print("DAMAGE-PER-ENERGY (DPE) ANALYSIS")
print("=" * 60)
# Work from the existing attack-level dataset
dpe_df = attack_df.copy()
# Ensure required numeric fields exist
dpe_df["Energy_Cost"] = pd.to_numeric(
    dpe_df["Energy_Cost"],
    errors="coerce"
)
dpe_df["Parsed_Damage"] = pd.to_numeric(
    dpe_df["Parsed_Damage"],
    errors="coerce"
)
# Keep attacks with valid positive energy cost and damage
dpe_valid = dpe_df[
    (dpe_df["Energy_Cost"] > 0) &
    dpe_df["Parsed_Damage"].notna()
].copy()
# Calculate Damage-per-Energy
dpe_valid["Damage_Per_Energy"] = (
    dpe_valid["Parsed_Damage"] /
    dpe_valid["Energy_Cost"]
)
print(f"Valid attacks for DPE analysis: {len(dpe_valid):,}")
# Summary statistics
dpe_stats = dpe_valid["Damage_Per_Energy"].describe()
print("\nDPE statistics:")
display(
    dpe_stats.to_frame(name="Damage_Per_Energy")
)
# Top attacks by DPE
top_dpe = (
    dpe_valid
    .sort_values("Damage_Per_Energy", ascending=False)
    .drop_duplicates(subset=["Card Name"])
    .head(15)
)
print("\nTop attacks by Damage-per-Energy:")
display(
    top_dpe[
        [
            "Card Name",
            "Move Name",
            "Energy_Cost",
            "Parsed_Damage",
            "Damage_Per_Energy"
        ]
    ].reset_index(drop=True)
)

# MATPLOTLIB VISUALIZATION
plot_dpe = top_dpe.sort_values(
    "Damage_Per_Energy",
    ascending=True
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_dpe["Card Name"],
    plot_dpe["Damage_Per_Energy"]
)
plt.title("Top Cards by Damage-per-Energy")
plt.xlabel("Damage per Energy")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Top Energy-Efficient Attacks

In [ ]:
print("=" * 60)
print("TOP ENERGY-EFFICIENT ATTACKS")
print("=" * 60)
# Rank individual attacks by Damage-per-Energy
top_dpe_attacks = (
    dpe_valid
    .sort_values(
        by="Damage_Per_Energy",
        ascending=False
    )
    .head(15)
    .copy()
)
print(f"Attacks analyzed: {len(dpe_valid):,}")
print("\nHighest Damage-per-Energy attacks:")
display(
    top_dpe_attacks[
        [
            "Card Name",
            "Move Name",
            "Energy_Cost",
            "Parsed_Damage",
            "Damage_Per_Energy"
        ]
    ]
    .reset_index(drop=True)
)

# MATPLOTLIB VISUALIZATION
plot_top_dpe = (
    top_dpe_attacks
    .sort_values("Damage_Per_Energy", ascending=True)
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_top_dpe["Card Name"],
    plot_top_dpe["Damage_Per_Energy"]
)
plt.title("Top Attacks by Damage-per-Energy")
plt.xlabel("Damage per Energy")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Energy Cost Distribution Analysis

In [ ]:
print("=" * 60)
print("ENERGY COST DISTRIBUTION ANALYSIS")
print("=" * 60)
energy_values = (
    dpe_valid["Energy_Cost"]
    .dropna()
    .astype(float)
)
print(f"Valid energy-cost records: {len(energy_values):,}")
print("\nEnergy cost statistics:")
display(
    energy_values.describe()
    .round(2)
    .to_frame(name="Energy_Cost")
)
# Frequency of each energy cost
energy_distribution = (
    energy_values
    .value_counts()
    .sort_index()
    .rename_axis("Energy_Cost")
    .reset_index(name="Attack_Count")
)
print("\nEnergy cost distribution:")
display(energy_distribution)
print("\nMost common energy cost:")
most_common_energy = energy_distribution.loc[
    energy_distribution["Attack_Count"].idxmax()
]
print(
    f"Energy Cost: {most_common_energy['Energy_Cost']:.0f} | "
    f"Attacks: {most_common_energy['Attack_Count']:,}"
)

# MATPLOTLIB VISUALIZATION
plt.figure(figsize=(10, 6))
plt.bar(
    energy_distribution["Energy_Cost"].astype(str),
    energy_distribution["Attack_Count"]
)
plt.title("Attack Distribution by Energy Cost")
plt.xlabel("Energy Cost")
plt.ylabel("Number of Attacks")
plt.tight_layout()
plt.show()

## Attack Efficiency Profile

In [ ]:
print("=" * 60)
print("ATTACK EFFICIENCY PROFILE")
print("=" * 60)
# Use the validated attack-level dataset from Cells 65–69
efficiency_df = dpe_valid.copy()
# Calculate normalized efficiency components
efficiency_df["Damage_Normalized"] = (
    efficiency_df["Parsed_Damage"] /
    efficiency_df["Parsed_Damage"].max()
)
efficiency_df["DPE_Normalized"] = (
    efficiency_df["Damage_Per_Energy"] /
    efficiency_df["Damage_Per_Energy"].max()
)
# Lower energy requirement is strategically preferable.
efficiency_df["Energy_Efficiency"] = (
    1 /
    efficiency_df["Energy_Cost"]
)
efficiency_df["Energy_Efficiency"] = (
    efficiency_df["Energy_Efficiency"] /
    efficiency_df["Energy_Efficiency"].max()
)
# Combined attack efficiency score
efficiency_df["Attack_Efficiency_Score"] = (
    0.40 * efficiency_df["Damage_Normalized"] +
    0.40 * efficiency_df["DPE_Normalized"] +
    0.20 * efficiency_df["Energy_Efficiency"]
)
print(f"Attacks evaluated: {len(efficiency_df):,}")
print("\nAttack efficiency statistics:")
display(
    efficiency_df[
        [
            "Parsed_Damage",
            "Energy_Cost",
            "Damage_Per_Energy",
            "Attack_Efficiency_Score"
        ]
    ]
    .describe()
    .round(4)
)
# Top individual attacks
top_efficiency = (
    efficiency_df
    .sort_values(
        "Attack_Efficiency_Score",
        ascending=False
    )
    .head(15)
    .copy()
)
print("\nTop attack efficiency profiles:")
display(
    top_efficiency[
        [
            "Card Name",
            "Move Name",
            "Parsed_Damage",
            "Energy_Cost",
            "Damage_Per_Energy",
            "Attack_Efficiency_Score"
        ]
    ]
    .reset_index(drop=True)
)

# MATPLOTLIB VISUALIZATION
plot_efficiency = (
    top_efficiency
    .sort_values("Attack_Efficiency_Score")
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_efficiency["Card Name"],
    plot_efficiency["Attack_Efficiency_Score"]
)
plt.title("Top Attack Efficiency Profiles")
plt.xlabel("Attack Efficiency Score")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Card Offensive Profile

In [ ]:
print("=" * 60)
print("CARD OFFENSIVE PROFILE")
print("=" * 60)
# Aggregate validated attack-level efficiency into card-level features
card_offensive_profile = (
    efficiency_df
    .groupby("Card Name")
    .agg(
        Max_Damage=("Parsed_Damage", "max"),
        Mean_Damage=("Parsed_Damage", "mean"),
        Max_DPE=("Damage_Per_Energy", "max"),
        Mean_DPE=("Damage_Per_Energy", "mean"),
        Mean_Energy_Cost=("Energy_Cost", "mean"),
        Best_Attack_Efficiency=("Attack_Efficiency_Score", "max"),
        Attack_Count=("Move Name", "count")
    )
    .reset_index()
)
# Normalize offensive features
card_offensive_profile["Damage_Score"] = (
    card_offensive_profile["Max_Damage"] /
    card_offensive_profile["Max_Damage"].max()
)
card_offensive_profile["DPE_Score"] = (
    card_offensive_profile["Max_DPE"] /
    card_offensive_profile["Max_DPE"].max()
)
card_offensive_profile["Efficiency_Score"] = (
    card_offensive_profile["Best_Attack_Efficiency"]
)
# Overall offensive score
card_offensive_profile["Offensive_Score"] = (
    0.40 * card_offensive_profile["Damage_Score"] +
    0.30 * card_offensive_profile["DPE_Score"] +
    0.30 * card_offensive_profile["Efficiency_Score"]
)
print(
    f"Cards with validated offensive data: "
    f"{len(card_offensive_profile):,}"
)
print("\nTop offensive cards:")
top_offensive_cards = (
    card_offensive_profile
    .sort_values(
        "Offensive_Score",
        ascending=False
    )
    .head(15)
)
display(
    top_offensive_cards[
        [
            "Card Name",
            "Max_Damage",
            "Mean_Damage",
            "Max_DPE",
            "Mean_Energy_Cost",
            "Attack_Count",
            "Offensive_Score"
        ]
    ]
    .reset_index(drop=True)
    .round(3)
)

# MATPLOTLIB VISUALIZATION
plot_offensive = (
    top_offensive_cards
    .sort_values("Offensive_Score")
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_offensive["Card Name"],
    plot_offensive["Offensive_Score"]
)
plt.title("Top Cards by Offensive Profile")
plt.xlabel("Offensive Score")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Utility / Effect Analysis

In [ ]:
print("=" * 60)
print("UTILITY / EFFECT ANALYSIS")
print("=" * 60)
# Build attack-level utility values from the existing effect data
utility_analysis = attack_df[
    ["Card Name", "Effect Explanation"]
].copy()
utility_analysis["Utility_Score"] = (
    utility_analysis["Effect Explanation"]
    .apply(calculate_utility_score)
)
# Aggregate utility information at card level
card_utility_profile = (
    utility_analysis
    .groupby("Card Name")
    .agg(
        Mean_Utility=("Utility_Score", "mean"),
        Max_Utility=("Utility_Score", "max"),
        Utility_Effects=("Utility_Score", lambda x: (x > 0).sum())
    )
    .reset_index()
)
# Overall card utility score
card_utility_profile["Utility_Profile_Score"] = (
    0.60 * card_utility_profile["Max_Utility"] +
    0.40 * card_utility_profile["Mean_Utility"]
)
print(
    f"Cards with utility data: "
    f"{len(card_utility_profile):,}"
)
print("\nTop utility-oriented cards:")
top_utility_cards = (
    card_utility_profile
    .sort_values(
        "Utility_Profile_Score",
        ascending=False
    )
    .head(15)
)
display(
    top_utility_cards[
        [
            "Card Name",
            "Mean_Utility",
            "Max_Utility",
            "Utility_Effects",
            "Utility_Profile_Score"
        ]
    ]
    .reset_index(drop=True)
    .round(4)
)

# MATPLOTLIB VISUALIZATION
plot_utility = (
    top_utility_cards
    .sort_values("Utility_Profile_Score")
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_utility["Card Name"],
    plot_utility["Utility_Profile_Score"]
)
plt.title("Top Cards by Utility Profile")
plt.xlabel("Utility Profile Score")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Strategic Combat Profile

In [ ]:
print("=" * 60)
print("STRATEGIC COMBAT PROFILE")
print("=" * 60)
# Combine the offensive and utility profiles
strategic_combat_profile = (
    card_offensive_profile[
        [
            "Card Name",
            "Offensive_Score",
            "Max_Damage",
            "Max_DPE",
            "Attack_Count"
        ]
    ]
    .merge(
        card_utility_profile[
            [
                "Card Name",
                "Utility_Profile_Score",
                "Max_Utility",
                "Utility_Effects"
            ]
        ],
        on="Card Name",
        how="outer"
    )
)
# Fill unavailable component values with neutral zero contribution
strategic_combat_profile[
    [
        "Offensive_Score",
        "Utility_Profile_Score",
        "Max_Damage",
        "Max_DPE",
        "Attack_Count",
        "Max_Utility",
        "Utility_Effects"
    ]
] = strategic_combat_profile[
    [
        "Offensive_Score",
        "Utility_Profile_Score",
        "Max_Damage",
        "Max_DPE",
        "Attack_Count",
        "Max_Utility",
        "Utility_Effects"
    ]
].fillna(0)

# Strategic combat score
strategic_combat_profile["Strategic_Combat_Score"] = (
    0.65 * strategic_combat_profile["Offensive_Score"] +
    0.35 * strategic_combat_profile["Utility_Profile_Score"]
)
print(
    f"Cards with strategic combat profiles: "
    f"{len(strategic_combat_profile):,}"
)
print("\nTop strategic combat profiles:")
top_strategic_cards = (
    strategic_combat_profile
    .sort_values(
        "Strategic_Combat_Score",
        ascending=False
    )
    .head(15)
)
display(
    top_strategic_cards[
        [
            "Card Name",
            "Offensive_Score",
            "Utility_Profile_Score",
            "Max_Damage",
            "Max_DPE",
            "Strategic_Combat_Score"
        ]
    ]
    .reset_index(drop=True)
    .round(4)
)

# MATPLOTLIB VISUALIZATION
plot_strategic = (
    top_strategic_cards
    .sort_values("Strategic_Combat_Score")
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_strategic["Card Name"],
    plot_strategic["Strategic_Combat_Score"]
)
plt.title("Top Cards by Strategic Combat Profile")
plt.xlabel("Strategic Combat Score")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Synergy + Combat Integration

In [ ]:
print("=" * 60)
print("SYNERGY + COMBAT INTEGRATION")
print("=" * 60)

# Aggregate each card's neutral synergy profile
synergy_profile = (
    synergy_pairs_df
    .groupby("Card_A")["Type_Neutral_Synergy"]
    .agg(
        Mean_Neutral_Synergy="mean",
        Max_Neutral_Synergy="max",
        Synergy_Connections="count"
    )
    .reset_index()
    .rename(columns={"Card_A": "Card Name"})
)

# Include cards that only appear as Card_B
synergy_profile_b = (
    synergy_pairs_df
    .groupby("Card_B")["Type_Neutral_Synergy"]
    .agg(
        Mean_Neutral_Synergy_B="mean",
        Max_Neutral_Synergy_B="max",
        Synergy_Connections_B="count"
    )
    .reset_index()
    .rename(columns={"Card_B": "Card Name"})
)

# Merge both directions
synergy_profile = (
    synergy_profile
    .merge(synergy_profile_b, on="Card Name", how="outer")
    .fillna(0)
)

# Combine directional measurements
synergy_profile["Mean_Neutral_Synergy"] = (
    synergy_profile["Mean_Neutral_Synergy"] +
    synergy_profile["Mean_Neutral_Synergy_B"]
) / (
    (synergy_profile["Synergy_Connections"] > 0).astype(int) +
    (synergy_profile["Synergy_Connections_B"] > 0).astype(int)
).replace(0, 1)
synergy_profile["Max_Neutral_Synergy"] = np.maximum(
    synergy_profile["Max_Neutral_Synergy"],
    synergy_profile["Max_Neutral_Synergy_B"]
)
synergy_profile["Synergy_Connections"] = (
    synergy_profile["Synergy_Connections"] +
    synergy_profile["Synergy_Connections_B"]
)

# Normalize synergy strength
synergy_profile["Synergy_Score"] = (
    synergy_profile["Mean_Neutral_Synergy"] /
    synergy_profile["Mean_Neutral_Synergy"].max()
)

# Combine combat strength and synergy
strategic_combat_synergy = (
    strategic_combat_profile
    .merge(
        synergy_profile[
            [
                "Card Name",
                "Mean_Neutral_Synergy",
                "Max_Neutral_Synergy",
                "Synergy_Connections",
                "Synergy_Score"
            ]
        ],
        on="Card Name",
        how="left"
    )
    .fillna(0)
)
strategic_combat_synergy["Combat_Synergy_Score"] = (
    0.70 * strategic_combat_synergy["Strategic_Combat_Score"] +
    0.30 * strategic_combat_synergy["Synergy_Score"]
)
print(
    f"Cards integrated: "
    f"{len(strategic_combat_synergy):,}"
)
print("\nTop combat + synergy profiles:")
top_integrated = (
    strategic_combat_synergy
    .sort_values(
        "Combat_Synergy_Score",
        ascending=False
    )
    .head(15)
)
display(
    top_integrated[
        [
            "Card Name",
            "Strategic_Combat_Score",
            "Mean_Neutral_Synergy",
            "Synergy_Connections",
            "Combat_Synergy_Score"
        ]
    ]
    .reset_index(drop=True)
    .round(4)
)

# MATPLOTLIB VISUALIZATION
plot_integrated = (
    top_integrated
    .sort_values("Combat_Synergy_Score")
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_integrated["Card Name"],
    plot_integrated["Combat_Synergy_Score"]
)
plt.title("Top Cards by Combat + Synergy Score")
plt.xlabel("Combat + Synergy Score")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Card Ranking / Deck Candidate Scoring

In [ ]:
print("=" * 60)
print("CARD RANKING / DECK CANDIDATE SCORING")
print("=" * 60)

# Build the final candidate score from the integrated profile
deck_candidate_scores = strategic_combat_synergy.copy()

# Rank cards using combat strength, synergy, and connection depth
deck_candidate_scores["Deck_Candidate_Score"] = (
    0.55 * deck_candidate_scores["Strategic_Combat_Score"] +
    0.30 * deck_candidate_scores["Synergy_Score"] +
    0.15 * (
        deck_candidate_scores["Synergy_Connections"] /
        deck_candidate_scores["Synergy_Connections"].max()
    )
)

# Rank all candidates
deck_candidate_scores["Candidate_Rank"] = (
    deck_candidate_scores["Deck_Candidate_Score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)
ranked_candidates = (
    deck_candidate_scores
    .sort_values(
        ["Deck_Candidate_Score", "Strategic_Combat_Score"],
        ascending=False
    )
    .reset_index(drop=True)
)
print(
    f"Cards ranked: "
    f"{len(ranked_candidates):,}"
)
print("\nTop deck candidates:")
display(
    ranked_candidates[
        [
            "Candidate_Rank",
            "Card Name",
            "Strategic_Combat_Score",
            "Synergy_Score",
            "Synergy_Connections",
            "Deck_Candidate_Score"
        ]
    ]
    .head(20)
    .round(4)
)

# MATPLOTLIB VISUALIZATION
plot_candidates = (
    ranked_candidates
    .head(15)
    .sort_values("Deck_Candidate_Score")
)
plt.figure(figsize=(11, 7))
plt.barh(
    plot_candidates["Card Name"],
    plot_candidates["Deck_Candidate_Score"]
)
plt.title("Top Deck Candidates")
plt.xlabel("Deck Candidate Score")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

## Top Candidate Comparison

In [ ]:
print("=" * 60)
print("TOP CANDIDATE COMPARISON")
print("=" * 60)
top_candidates = (
    ranked_candidates
    .head(12)
    .copy()
)
print(f"Candidates compared: {len(top_candidates)}")
display(
    top_candidates[
        [
            "Candidate_Rank",
            "Card Name",
            "Strategic_Combat_Score",
            "Synergy_Score",
            "Synergy_Connections",
            "Deck_Candidate_Score"
        ]
    ]
    .reset_index(drop=True)
    .round(4)
)

# COMPARATIVE VISUALIZATION
plot_df = (
    top_candidates
    .sort_values("Deck_Candidate_Score")
)
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(
    plot_df["Card Name"],
    plot_df["Deck_Candidate_Score"],
    label="Final Candidate Score"
)
ax.scatter(
    plot_df["Strategic_Combat_Score"],
    plot_df["Card Name"],
    s=70,
    label="Combat Score"
)
ax.scatter(
    plot_df["Synergy_Score"],
    plot_df["Card Name"],
    s=70,
    label="Synergy Score"
)
ax.set_title(
    "Top Deck Candidates: Combat Strength vs Synergy",
    fontsize=15
)
ax.set_xlabel("Score")
ax.set_ylabel("Card Name")
ax.legend()
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## Recommended Card Combinations / Synergy Network

In [ ]:
print("=" * 60)
print("RECOMMENDED CARD COMBINATIONS / SYNERGY NETWORK")
print("=" * 60)

# Focus on the strongest candidates from Cell 76
candidate_names = top_candidates["Card Name"].tolist()

# Extract synergy relationships among top candidates
candidate_synergy = synergy_pairs_df[
    synergy_pairs_df["Card_A"].isin(candidate_names) &
    synergy_pairs_df["Card_B"].isin(candidate_names)
].copy()
print(f"Top candidates analyzed: {len(candidate_names)}")
print(f"Candidate-to-candidate synergy links: {len(candidate_synergy):,}")

# Build an undirected pair table
pair_records = []
for _, row in candidate_synergy.iterrows():
    card_a = row["Card_A"]
    card_b = row["Card_B"]

    # Avoid duplicate A-B / B-A relationships
    pair_key = tuple(sorted([card_a, card_b]))
    pair_records.append(
        (
            pair_key[0],
            pair_key[1],
            row["Type_Neutral_Synergy"]
        )
    )
pair_df = pd.DataFrame(
    pair_records,
    columns=[
        "Card_A",
        "Card_B",
        "Synergy_Score"
    ]
)
if not pair_df.empty:
    pair_df = (
        pair_df
        .groupby(
            ["Card_A", "Card_B"],
            as_index=False
        )["Synergy_Score"]
        .max()
        .sort_values(
            "Synergy_Score",
            ascending=False
        )
    )
print("\nStrongest candidate combinations:")
display(
    pair_df
    .head(15)
    .reset_index(drop=True)
    .round(4)
)

# SYNERGY MATRIX
synergy_matrix = pd.DataFrame(
    np.nan,
    index=candidate_names,
    columns=candidate_names
)
np.fill_diagonal(
    synergy_matrix.values,
    1.0
)
for _, row in pair_df.iterrows():
    synergy_matrix.loc[
        row["Card_A"],
        row["Card_B"]
    ] = row["Synergy_Score"]
    synergy_matrix.loc[
        row["Card_B"],
        row["Card_A"]
    ] = row["Synergy_Score"]

# MATPLOTLIB VISUALIZATION
plt.figure(figsize=(12, 10))
matrix_values = synergy_matrix.astype(float)
image = plt.imshow(
    matrix_values,
    aspect="auto",
    interpolation="nearest"
)
plt.colorbar(
    image,
    label="Neutral Synergy Score"
)
plt.xticks(
    range(len(candidate_names)),
    candidate_names,
    rotation=75,
    ha="right"
)
plt.yticks(
    range(len(candidate_names)),
    candidate_names
)
plt.title(
    "Synergy Network Among Top Deck Candidates",
    fontsize=15
)
plt.xlabel("Candidate Card")
plt.ylabel("Candidate Card")
plt.tight_layout()
plt.show()

# Final Strategy Dashboard / Key Findings

In [ ]:
print("=" * 60)
print("FINAL STRATEGY DASHBOARD")
print("=" * 60)

# Key metrics
best_card = ranked_candidates.iloc[0]
print("\nKEY STRATEGY METRICS")
print("-" * 60)
print(f"Cards evaluated:              {len(ranked_candidates):,}")
print(f"Top deck candidate:            {best_card['Card Name']}")
print(f"Top candidate score:           {best_card['Deck_Candidate_Score']:.4f}")
print(f"Strategic combat score:        {best_card['Strategic_Combat_Score']:.4f}")
print(f"Synergy score:                 {best_card['Synergy_Score']:.4f}")
print(f"Synergy connections:           {int(best_card['Synergy_Connections'])}")

# Dashboard data
dashboard_df = (
    ranked_candidates
    .head(10)
    .copy()
    .sort_values("Deck_Candidate_Score")
)

# VISUALIZATION 1 — FINAL CANDIDATE RANKING
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    dashboard_df["Card Name"],
    dashboard_df["Deck_Candidate_Score"]
)
ax.set_title(
    "Top Deck Candidates — Final Ranking",
    fontsize=16
)
ax.set_xlabel("Deck Candidate Score")
ax.set_ylabel("Card")
for bar, value in zip(
    bars,
    dashboard_df["Deck_Candidate_Score"]
):
    ax.text(
        bar.get_width() + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center"
    )
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# VISUALIZATION 2 — COMBAT VS SYNERGY
plt.figure(figsize=(11, 7))
scatter = plt.scatter(
    ranked_candidates["Strategic_Combat_Score"],
    ranked_candidates["Synergy_Score"],
    s=45,
    alpha=0.65
)

# Highlight top candidates
plt.scatter(
    dashboard_df["Strategic_Combat_Score"],
    dashboard_df["Synergy_Score"],
    s=90,
    edgecolors="black",
    linewidths=0.8,
    label="Top 10 Candidates"
)
plt.title(
    "Strategic Combat Strength vs Synergy",
    fontsize=16
)
plt.xlabel("Strategic Combat Score")
plt.ylabel("Synergy Score")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# VISUALIZATION 3 — SCORE COMPONENT BREAKDOWN
component_df = (
    dashboard_df[
        [
            "Card Name",
            "Strategic_Combat_Score",
            "Synergy_Score",
            "Deck_Candidate_Score"
        ]
    ]
    .set_index("Card Name")
)
component_df.plot(
    kind="bar",
    figsize=(13, 7)
)
plt.title(
    "Top Candidates — Strategic Score Components",
    fontsize=16
)
plt.xlabel("Card")
plt.ylabel("Score")
plt.xticks(
    rotation=45,
    ha="right"
)
plt.legend(
    title="Score Component"
)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()
print("\n" + "=" * 60)
print("STRATEGY DASHBOARD COMPLETE")
print("=" * 60)

# Final Submission Visualizations

In [ ]:
print("\nGenerating final submission visualizations...")

# 1. FINAL SCORE DISTRIBUTION
score_series = (
    ranked_candidates["Deck_Candidate_Score"]
    .dropna()
)
plt.figure(figsize=(11, 6))
sns.histplot(
    score_series,
    bins=25,
    kde=True
)
plt.axvline(
    score_series.mean(),
    linestyle="--",
    linewidth=2,
    label=f"Mean Score: {score_series.mean():.2f}"
)
plt.title(
    "Final Deck Candidate Score Distribution",
    fontsize=16
)
plt.xlabel("Deck Candidate Score")
plt.ylabel("Number of Cards")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# 2. TOP 15 FINAL CANDIDATES
top_15 = (
    ranked_candidates
    .sort_values(
        "Deck_Candidate_Score",
        ascending=False
    )
    .head(15)
    .copy()
    .sort_values("Deck_Candidate_Score")
)
plt.figure(figsize=(12, 8))
bars = plt.barh(
    top_15["Card Name"],
    top_15["Deck_Candidate_Score"]
)
plt.title(
    "Top 15 PTCG AI Deck Candidates",
    fontsize=16
)
plt.xlabel("Final Deck Candidate Score")
plt.ylabel("Card Name")
for bar, value in zip(
    bars,
    top_15["Deck_Candidate_Score"]
):
    plt.text(
        bar.get_width() + 0.15,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f}",
        va="center",
        fontsize=10
    )
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# 3. STRATEGY COMPONENT HEATMAP
heatmap_df = (
    ranked_candidates
    .sort_values(
        "Deck_Candidate_Score",
        ascending=False
    )
    .head(12)
    [
        [
            "Card Name",
            "Strategic_Combat_Score",
            "Synergy_Score",
            "Deck_Candidate_Score"
        ]
    ]
    .set_index("Card Name")
)
plt.figure(figsize=(11, 8))
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cmap="YlGnBu",
    cbar_kws={"label": "Score"}
)
plt.title(
    "Top Candidates — Strategic Score Profile",
    fontsize=16
)
plt.xlabel("Strategy Component")
plt.ylabel("Card Name")
plt.tight_layout()
plt.show()

# 4. SELF-CONTAINED VALIDATION
required_columns = [
    "Card Name",
    "Strategic_Combat_Score",
    "Synergy_Score",
    "Synergy_Connections",
    "Deck_Candidate_Score"
]
missing_columns = [
    column
    for column in required_columns
    if column not in ranked_candidates.columns
]
validation_results = {
    "Required columns present": len(missing_columns) == 0,
    "No missing candidate scores": (
        ranked_candidates["Deck_Candidate_Score"]
        .notna()
        .all()
    ),
    "No missing strategic scores": (
        ranked_candidates["Strategic_Combat_Score"]
        .notna()
        .all()
    ),
    "No missing synergy scores": (
        ranked_candidates["Synergy_Score"]
        .notna()
        .all()
    ),
    "Unique card rankings": (
        ranked_candidates["Card Name"]
        .duplicated()
        .sum() == 0
    )
}
validation_plot = pd.Series(
    validation_results,
    dtype=int
)

# 5. VALIDATION VISUALIZATION
plt.figure(figsize=(12, 4))
sns.heatmap(
    validation_plot.to_frame().T,
    annot=True,
    fmt="d",
    cmap="RdYlGn",
    cbar=False,
    linewidths=1,
    xticklabels=validation_plot.index,
    yticklabels=["Validation"]
)
plt.title(
    "Final Engine Validation Status",
    fontsize=16
)
plt.xticks(
    rotation=35,
    ha="right"
)
plt.tight_layout()
plt.show()

# FINAL STATUS
print("\n" + "=" * 60)
print("FINAL VISUAL VALIDATION COMPLETE")
print("=" * 60)
if all(validation_results.values()):
    print("ALL VALIDATION CHECKS PASSED")
    print("Final analytical outputs are internally validated.")
else:
    print("REVIEW REQUIRED")
    print(f"Missing columns: {missing_columns}")
print("=" * 60)

# Notebook Completion & Future Development

## Current Version

This notebook establishes the current analytical foundation of the **PTCG-AI Deck Intelligence Strategy Engine**.

The completed workflow covers:

**Data Preparation → Feature Engineering → Attack Analysis → DPE → Strategic Card Intelligence → Utility Analysis → Synergy Modeling → Type-Neutral Synergy → Combat + Synergy Integration → Candidate Ranking → Visualization → Validation**

The current results provide an interpretable baseline for evaluating card strength and strategic compatibility within the competition card pool.

## Future Development

The engine is intentionally designed for continued development.

Future iterations can extend the current framework toward:

* Full deck-level optimization
* Legal deck construction and validation
* Multi-card combination optimization
* Energy-resource planning
* Trainer and support-card intelligence
* Turn-by-turn battle simulation
* Opponent-aware strategy
* Game-state evaluation
* Probabilistic decision modeling
* Machine-learning-based deck and action prediction
* Interactive strategy dashboards
* Advanced synergy-network modeling
* Robustness and benchmark evaluation

## Development Principle

Future improvements should preserve validated components while extending the strategic intelligence layer.

Each new capability should follow:

**Develop → Test → Validate → Visualize → Integrate**

This notebook therefore serves as the current **baseline version of the PTCG-AI Deck Intelligence Strategy Engine**, providing a structured foundation for further research, experimentation, optimization, and AI-driven strategic development.


In [ ]:
# ============================================================
# COMPETITION SUBMISSION FILE EXPORT
# ============================================================

print("=" * 60)
print("CREATING COMPETITION SUBMISSION FILE")
print("=" * 60)

# Export the final ranked card intelligence results.
# This creates a physical output file for competition submission.
submission_columns = [
    "Candidate_Rank",
    "Card Name",
    "Strategic_Combat_Score",
    "Synergy_Score",
    "Synergy_Connections",
    "Deck_Candidate_Score"
]

submission_df = (
    ranked_candidates[submission_columns]
    .copy()
    .sort_values("Candidate_Rank")
    .reset_index(drop=True)
)

# Competition environments such as Kaggle expose /kaggle/working
# as the notebook's writable output directory.
output_dir = "/kaggle/working"
if not os.path.exists(output_dir):
    output_dir = os.getcwd()

submission_path = os.path.join(
    output_dir,
    "submission.csv"
)

submission_df.to_csv(
    submission_path,
    index=False
)

print(f"Submission file created: {submission_path}")
print(f"Rows exported: {len(submission_df):,}")
print("Columns exported:")
print(", ".join(submission_df.columns))

print("\nTop 10 submission records:")
display(submission_df.head(10).round(4))

print("\n" + "=" * 60)
print("SUBMISSION FILE READY")
print("=" * 60)
